# Project 5.1 — Home Loan Data Analysis (Deep Learning)
**Predict whether an applicant will repay a home loan** — binary classification on the
Home Credit dataset (307,511 rows x 122 cols), which is **highly imbalanced (~8% minority)**.

This notebook follows `plan.md` and `explanation.md` in the same project folder.
Run the cells top-to-bottom. See the loader cell for how to supply the CSV in Colab.

> ⚠️ **Target meaning:** the bundled data dictionary says `TARGET 1 = repaid`, but this is the
> standard Home Credit set where `1 = default`. The ~8% rate of 1s matches "default". The code
> works either way; just state your interpretation in your conclusion.

## 1. Project Overview & Business Problem
Goal: flag risky loans before approval. Because defaults are rare, we optimise **AUC / recall**, not accuracy.

## 2. Importing Required Libraries

In [ ]:
# Colab already has these, but this guarantees versions are present.
# (imbalanced-learn is only needed if you try the optional SMOTE path.)
!pip -q install imbalanced-learn >/dev/null 2>&1
print("setup cell done")

In [ ]:
import os, numpy as np, pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import (classification_report, confusion_matrix,
                             roc_auc_score, roc_curve, precision_recall_curve,
                             average_precision_score, f1_score)
from sklearn.utils.class_weight import compute_class_weight

# Reproducibility
SEED = 42
np.random.seed(SEED); tf.random.set_seed(SEED)
pd.set_option('display.max_columns', 200)
sns.set(style='whitegrid')
print('TensorFlow', tf.__version__)

## 3. Loading the Dataset
Upload `loan_data (1).csv`, or mount Google Drive and point `path` to it.

In [ ]:
# ---- Flexible data loader (works in Colab or locally) ----
FILENAME = 'loan_data (1).csv'   # Home Credit application data (~166 MB)

CANDIDATES = [
    FILENAME,
    '/content/' + FILENAME,
    'CEP_1_Datasets/' + FILENAME,
    '/content/drive/MyDrive/' + FILENAME,
]
path = next((p for p in CANDIDATES if os.path.exists(p)), None)

if path is None:
    try:
        from google.colab import files
        print('Please upload', FILENAME, '(or mount Google Drive instead).')
        uploaded = files.upload()
        path = list(uploaded.keys())[0]
    except Exception:
        raise FileNotFoundError(
            'Put the CSV next to the notebook, or mount Drive:\n'
            '  from google.colab import drive; drive.mount("/content/drive")\n'
            'then set path = "/content/drive/MyDrive/.../loan_data (1).csv"')

df = pd.read_csv(path)
print('Loaded:', path)
print('Shape:', df.shape)
df.head()

## 4. Data Understanding

In [ ]:
print('Rows, Cols:', df.shape)
print('\nTarget distribution (TARGET):')
print(df['TARGET'].value_counts())
print(df['TARGET'].value_counts(normalize=True).round(4))

print('\nDtypes:')
print(df.dtypes.value_counts())

print('\nTop missing columns:')
print((df.isnull().mean().sort_values(ascending=False).head(15)).round(3))

## 5. Data Cleaning & Quality Checks
Drop the ID, fix the `365243` sentinel, fix `XNA` gender, drop >50%-missing columns.

In [ ]:
# --- 1) Drop the random ID (no predictive value) ---
df = df.drop(columns=['SK_ID_CURR'])

# --- 2) Fix the DAYS_EMPLOYED sentinel 365243 (means 'not employed') ---
df['DAYS_EMPLOYED'] = df['DAYS_EMPLOYED'].replace(365243, np.nan)

# --- 3) Fix invalid category CODE_GENDER == 'XNA' ---
if 'CODE_GENDER' in df.columns:
    df['CODE_GENDER'] = df['CODE_GENDER'].replace('XNA', np.nan)

# --- 4) Drop columns with > 50% missing (mostly empty building-info cols) ---
missing = df.isnull().mean()
high_missing = missing[missing > 0.50].index.tolist()
df = df.drop(columns=high_missing)
print('Dropped %d high-missing columns.' % len(high_missing))
print('Shape after cleaning:', df.shape)

## 6-8. Exploratory Data Analysis (Univariate -> Bivariate)

In [ ]:
plt.figure(figsize=(5,4))
sns.countplot(x='TARGET', data=df)
plt.title('Class balance (0 = majority, 1 = minority ~8%)')
plt.show()
print('Minority share: %.2f%%' % (100*df['TARGET'].mean()))

In [ ]:
# A few quick EDA views (expand as you like)
num_preview = ['AMT_INCOME_TOTAL', 'AMT_CREDIT', 'DAYS_BIRTH', 'EXT_SOURCE_2']
num_preview = [c for c in num_preview if c in df.columns]

fig, axes = plt.subplots(1, len(num_preview), figsize=(5*len(num_preview), 4))
for ax, col in zip(np.atleast_1d(axes), num_preview):
    sns.histplot(df[col].dropna(), bins=40, ax=ax)
    ax.set_title(col)
plt.tight_layout(); plt.show()

# Age in years is more readable than DAYS_BIRTH
if 'DAYS_BIRTH' in df.columns:
    (df['DAYS_BIRTH'] / -365).describe()

In [ ]:
# Bivariate: default rate by a key categorical
if 'NAME_EDUCATION_TYPE' in df.columns:
    rate = df.groupby('NAME_EDUCATION_TYPE')['TARGET'].mean().sort_values()
    plt.figure(figsize=(7,4))
    rate.plot(kind='barh')
    plt.title('Default rate by education'); plt.xlabel('mean TARGET'); plt.show()

## 11. Feature & Target Separation

In [ ]:
# Separate target and features
y = df['TARGET'].astype(int)
X = df.drop(columns=['TARGET'])
print('X:', X.shape, '| y:', y.shape)

### Impute missing values (median / most-frequent)

In [ ]:
# Impute missing values (median for numeric, most-frequent for categorical)
num_cols = X.select_dtypes(include=[np.number]).columns.tolist()
cat_cols = X.select_dtypes(include=['object']).columns.tolist()
print('numeric:', len(num_cols), '| categorical:', len(cat_cols))

if num_cols:
    X[num_cols] = SimpleImputer(strategy='median').fit_transform(X[num_cols])
if cat_cols:
    X[cat_cols] = SimpleImputer(strategy='most_frequent').fit_transform(X[cat_cols])

print('Missing left:', int(X.isnull().sum().sum()))

## 9. Feature Transformation (Task 1) — encode categoricals
Rare levels grouped into `Other`; nominal columns one-hot encoded with `drop_first`.

In [ ]:
# --- Feature Transformation (Task 1): categorical -> numeric ---
# Group rare categories (<1%) into 'Other' to keep one-hot width manageable.
for col in cat_cols:
    freq = X[col].value_counts(normalize=True)
    rare = freq[freq < 0.01].index
    X[col] = X[col].where(~X[col].isin(rare), other='Other')

X = pd.get_dummies(X, columns=cat_cols, drop_first=True)
X = X.astype(np.float32)
print('Shape after one-hot encoding:', X.shape)

## 10. Correlation Analysis & Feature Selection (Task 3)
Drop one of each pair with |corr| > 0.90 (removes redundant `_AVG/_MODE/_MEDI`-style duplicates).

In [ ]:
# --- Correlation Analysis & Feature Selection (Task 3) ---
corr = X.corr().abs()
upper = corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))
to_drop = [c for c in upper.columns if (upper[c] > 0.90).any()]
X = X.drop(columns=to_drop)
print('Dropped %d highly-correlated columns (|corr|>0.90).' % len(to_drop))
print('Final feature count:', X.shape[1])

## 12. Train / Validation / Test Split
3-way, stratified so the ~8% ratio is preserved everywhere.

In [ ]:
# --- Train / Validation / Test split (stratified) ---
X_trainval, X_test, y_trainval, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=SEED)
X_train, X_val, y_train, y_val = train_test_split(
    X_trainval, y_trainval, test_size=0.20, stratify=y_trainval, random_state=SEED)

print('train:', X_train.shape, '| val:', X_val.shape, '| test:', X_test.shape)

## 13. Preprocessing & Scaling
**Fit the scaler on train only** — transforming val/test with train stats avoids leakage.

In [ ]:
# --- Scale: FIT ON TRAIN ONLY, then transform all (no leakage) ---
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_val   = scaler.transform(X_val)
X_test  = scaler.transform(X_test)
print('Scaled. n_features =', X_train.shape[1])

## 14. Class Imbalance Handling
Balanced class weights make the optimiser pay attention to the rare class (no invented data).

In [ ]:
# --- Class imbalance handling via class weights (train only) ---
classes = np.unique(y_train)
weights = compute_class_weight('balanced', classes=classes, y=y_train)
class_weight = dict(zip(classes, weights))
print('class_weight =', class_weight)

## 15. Deep Learning Model Architecture

In [ ]:
# --- Deep Learning Model Architecture (Keras Sequential) ---
n_features = X_train.shape[1]

model = keras.Sequential([
    layers.Input(shape=(n_features,)),
    layers.Dense(128, activation='relu'),
    layers.BatchNormalization(),
    layers.Dropout(0.3),
    layers.Dense(64, activation='relu'),
    layers.BatchNormalization(),
    layers.Dropout(0.3),
    layers.Dense(32, activation='relu'),
    layers.Dense(1, activation='sigmoid'),   # binary -> 1 neuron + sigmoid
])

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss='binary_crossentropy',
    metrics=['accuracy',
             keras.metrics.AUC(name='auc'),
             keras.metrics.Precision(name='precision'),
             keras.metrics.Recall(name='recall')],
)
model.summary()

## 16-17. Model Compilation, Training & Overfitting Analysis

In [ ]:
# --- Train ---
callbacks = [
    keras.callbacks.EarlyStopping(monitor='val_auc', mode='max',
                                  patience=6, restore_best_weights=True),
    keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3),
]

history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=50,
    batch_size=512,
    class_weight=class_weight,
    callbacks=callbacks,
    verbose=1,
)

In [ ]:
# --- Training history / overfitting check ---
h = history.history
fig, ax = plt.subplots(1, 2, figsize=(12,4))
ax[0].plot(h['loss'], label='train'); ax[0].plot(h['val_loss'], label='val')
ax[0].set_title('Loss'); ax[0].legend()
ax[1].plot(h['auc'], label='train'); ax[1].plot(h['val_auc'], label='val')
ax[1].set_title('AUC'); ax[1].legend()
plt.show()

## 18-19. Model Evaluation & Threshold Selection (on validation)

In [ ]:
# --- Evaluate on VALIDATION and pick the best threshold (by F1) ---
val_proba = model.predict(X_val).ravel()
print('Validation ROC-AUC: %.4f' % roc_auc_score(y_val, val_proba))

prec, rec, thr = precision_recall_curve(y_val, val_proba)
f1s = 2*prec*rec / (prec+rec+1e-9)
best_idx = int(np.nanargmax(f1s))
best_threshold = thr[best_idx] if best_idx < len(thr) else 0.5
print('Best threshold (max F1 on val): %.3f  ->  F1=%.3f' % (best_threshold, f1s[best_idx]))

val_pred = (val_proba >= best_threshold).astype(int)
print('\nValidation report @ tuned threshold:')
print(classification_report(y_val, val_pred, digits=3))

## 20. Final Test Evaluation
Apply the locked threshold once to the held-out test set — this is the honest score.

In [ ]:
# --- FINAL TEST EVALUATION (held-out, done once) ---
test_proba = model.predict(X_test).ravel()
test_pred  = (test_proba >= best_threshold).astype(int)

print('TEST ROC-AUC : %.4f' % roc_auc_score(y_test, test_proba))
print('TEST PR-AUC  : %.4f' % average_precision_score(y_test, test_proba))
print('TEST Accuracy: %.4f' % (test_pred == y_test).mean())
print('\nClassification report (TEST):')
print(classification_report(y_test, test_pred, digits=3))

cm = confusion_matrix(y_test, test_pred)
plt.figure(figsize=(4.5,4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.xlabel('Predicted'); plt.ylabel('Actual'); plt.title('Confusion Matrix (TEST)')
plt.show()

# ROC curve
fpr, tpr, _ = roc_curve(y_test, test_proba)
plt.figure(figsize=(5,4))
plt.plot(fpr, tpr, label='ROC (AUC=%.3f)' % roc_auc_score(y_test, test_proba))
plt.plot([0,1],[0,1],'--', color='grey')
plt.xlabel('FPR'); plt.ylabel('TPR'); plt.title('ROC curve (TEST)'); plt.legend()
plt.show()

## 21. Save the Model & Pipeline

In [ ]:
# --- Save the model (and optionally download it) ---
model.save('home_loan_model.keras')
print('Saved home_loan_model.keras')
# In Colab you can download it:
# from google.colab import files; files.download('home_loan_model.keras')

## 22. Business Insights, Limitations & Conclusion
- **Read AUC and recall**, not accuracy: predicting "all repaid" already scores ~92%.
- Strongest signals are typically `EXT_SOURCE_*` and the credit/income amounts.
- **Limitations:** single application table only (no bureau/history tables); class weights
  trade precision for recall. **Next steps:** try SMOTE, add engineered ratios, or benchmark
  against LightGBM/XGBoost.